This notebook contains code for analyzing statistics of the AS2Biz and AS2Web datasets, and for producing the numbers reported in our paper.

In [25]:
import json


# Load AS2Biz
with open("./datasets/2025-01/as2biz.json", "r") as f:
    as2biz = json.load(f)

# Load AS2Web
with open("./datasets/2025-01/as2web.json", "r", encoding="latin-1") as f:
    as2web = json.load(f)

# Load AS scope: 
# Includes ASes that were in "assigned" status based on RIR delegation files as of 2025-01-01,
# and were active in the BGP routing table during 2024.
with open("./datasets/2025-01/as_scope.json", "r") as f:
    scope = json.load(f)

Statistics of AS2Web (Table 1)

In [22]:
whois_total_as, ipinfo_total_as, pdb_total_as, perplexity_total_as = [], [], [], []
whois_accessible_as, ipinfo_accessible_as, pdb_accessible_as, perplexity_accessible_as = [], [], [], []

for asn, data in as2web.items():
    url = data["Website"]
    sources = data["Source"]
    accessible = data["Accessible"]

    for source in sources:
        if source == "Whois":
            whois_total_as.append(asn)
            if accessible:
                whois_accessible_as.append(asn)
        elif source == "IPinfo":
            ipinfo_total_as.append(asn)
            if accessible:
                ipinfo_accessible_as.append(asn)
        elif source == "PeeringDB":
            pdb_total_as.append(asn)
            if accessible:
                pdb_accessible_as.append(asn)
        elif source == "Perplexity AI sonar-pro":
            perplexity_total_as.append(asn)
            if accessible:
                perplexity_accessible_as.append(asn)

print(f"Total ASes in scope: {len(scope)}")
print(f"AS-centered sources: {len(set(whois_total_as+ipinfo_total_as+pdb_total_as))} ASes, {len(set(whois_accessible_as+ipinfo_accessible_as+pdb_accessible_as))} accessible.")
print(f" |-Whois-based: {len(whois_total_as)} ASes, {len(whois_accessible_as)} accessible.")
print(f" |-IPinfo-based: {len(ipinfo_total_as)} ASes, {len(ipinfo_accessible_as)} accessible.")
print(f" \-PeeringDB-based: {len(pdb_total_as)} ASes, {len(pdb_accessible_as)} accessible.")
print(f"Perplexity AI-based: {len(perplexity_total_as)} ASes, {len(perplexity_accessible_as)} accessible.")
print(f"{len(set(scope) - set(as2web))} ASes in scope not found in AS2Web.")


Total ASes in scope: 83808
AS-centered sources: 76162 ASes, 72103 accessible.
 |-Whois-based: 59901 ASes, 57075 accessible.
 |-IPinfo-based: 63713 ASes, 60445 accessible.
 \-PeeringDB-based: 20483 ASes, 19688 accessible.
Perplexity AI-based: 6426 ASes, 5750 accessible.
1220 ASes in scope not found in AS2Web.


Statistics of AS2Biz (Table 2)

In [29]:
only_direct_as, only_inherited_as, direct_inherited_as = [], [], []
wikipedia_as, crunchbase_as, perplexity_ai_as = [], [], []

for asn, data in as2biz.items():
    if_direct, if_inherited = False, False
    if_wikipedia, if_crunchbase, if_perplexity_ai = False, False, False

    for category, source in data.items():
        if source == "Direct - Website":
            if_direct = True
        if "Inherit" in source:
            if_inherited = True
        if "Wikipedia" in source:
            if_wikipedia = True
        if "Crunchbase" in source:
            if_crunchbase = True
        if "Perplexity AI" in source:
            if_perplexity_ai = True

    if if_direct and not if_inherited:
        only_direct_as.append(asn)
    elif not if_direct and if_inherited:
        only_inherited_as.append(asn)
    elif if_direct and if_inherited:
        direct_inherited_as.append(asn)
    else:
        if if_wikipedia:
            wikipedia_as.append(asn)
        if if_crunchbase:
            crunchbase_as.append(asn)
        if if_perplexity_ai:
            perplexity_ai_as.append(asn)
        if not (if_wikipedia or if_crunchbase or if_perplexity_ai):
            print(asn, as2biz[asn])
            raise ValueError("AS does not belong to any category.")

print(f"Total ASes in scope: {len(scope)}")
print(f"Main web-based approach: {len(set(only_direct_as + only_inherited_as + direct_inherited_as))} ASes.")
print(f" |-Only direct: {len(only_direct_as)} ASes.")
print(f" |-Only inherited: {len(only_inherited_as)} ASes.")
print(f" \\-Both direct and inherited: {len(direct_inherited_as)} ASes.")
print(f"Fallback methods: {len(set(wikipedia_as + crunchbase_as + perplexity_ai_as))} ASes.")
print(f" |-Wikipedia-based: {len(wikipedia_as)} ASes.")
print(f" |-Crunchbase-based: {len(crunchbase_as)} ASes.")
print(f" \\-Perplexity AI-based: {len(perplexity_ai_as)} ASes.")
print(f"{len(set(as2biz))} ASes in scope found in AS2Biz.")
print(f"{len(set(scope) - set(as2biz))} ASes in scope not found in AS2Biz.")


Total ASes in scope: 83808
Main web-based approach: 71604 ASes.
 |-Only direct: 68058 ASes.
 |-Only inherited: 996 ASes.
 \-Both direct and inherited: 2550 ASes.
Fallback methods: 7444 ASes.
 |-Wikipedia-based: 876 ASes.
 |-Crunchbase-based: 2101 ASes.
 \-Perplexity AI-based: 4467 ASes.
79048 ASes in scope found in AS2Biz.
4760 ASes in scope not found in AS2Biz.
